In [9]:
import torch
from torch.utils import data
import re
from torch import nn
from torch.nn import functional as F
import math
import random
batch_size=32
num_step=35
lr=1
epoch=500

In [10]:
def corpus_vocab_maker():
    with open('../data/timemachine.txt','r') as file:
        text=file.read().lower()
    text=re.sub('[^a-zA-Z]+',' ',text).strip()
    text=text.split()
    idx_to_token=[*(['<unk>']+list(set(text)))]
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab={
        'idx_to_token':idx_to_token,
        'token_to_idx':token_to_idx

    }
    corpus=torch.tensor([token_to_idx[token] for token in text])
    return corpus,vocab

In [11]:
def train_iter_maker(corpus,batch_size,num_step):
    t = random.randint(
        0,
        num_step - 1
    )
    corpus = corpus[t:]
    sequence=corpus.unfold(dimension=0,size=num_step+1,step=num_step)
    x = sequence[:, :-1]
    y = sequence[:, 1:]
    dataset=data.TensorDataset(x,y)
    train_iter=data.DataLoader(dataset,batch_size=batch_size,shuffle=True)
    return train_iter


In [12]:
corpus,vocab=corpus_vocab_maker()
vocab_size=len(vocab['token_to_idx'])
device=torch.device('cuda:0')



In [13]:
class lstm_net(nn.Module):
    def __init__(self,input_num,hidden_layer):
        super().__init__()
        self.input_num=input_num
        self.hidden_layer=hidden_layer
        self.lstm=nn.LSTM(input_size=input_num,hidden_size=hidden_layer,num_layers=2)
        self.output=nn.Linear(hidden_layer,input_num)
    def forward(self,x,state):
        x=F.one_hot(x.T.long(),num_classes=self.input_num).float()
        lstm_result,state=self.lstm(x,state)
        lstm_result=lstm_result.reshape(-1,self.hidden_layer)
        result=self.output(lstm_result)
        return result,state
    def get_new_state(self,batch_size):
        l1=torch.zeros(self.lstm.num_layers,batch_size,self.hidden_layer).to(device)
        l2=torch.zeros(self.lstm.num_layers,batch_size,self.hidden_layer).to(device)
        return l1,l2
    



In [14]:
net=lstm_net(vocab_size,256).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(trainer,T_max=epoch,eta_min=1e-4)
loss=nn.CrossEntropyLoss()
for i in range(epoch):
    net.train()
    train_iter=train_iter_maker(corpus,batch_size,num_step)
    train_loss=0.0
    train_count=0
    for x,y in train_iter:
        x=x.to(device)
        y=y.T.reshape(-1).to(device)
        state=net.get_new_state(x.shape[0])
        logit,state=net(x,state)
        trainer.zero_grad()
        l=loss(logit,y)
        l.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(),max_norm=1.0)
        trainer.step()
        train_loss+=l.item()*len(y)
        train_count+=len(y)
    scheduler.step()
    print(f"第{i+1}epoch的loss:{train_loss/train_count},困惑度：{math.exp(train_loss/train_count)}")


第1epoch的loss:7.807758223691726,困惑度：2459.6103539844808
第2epoch的loss:6.741635078038925,困惑度：846.9444245520358
第3epoch的loss:6.57302143634894,困惑度：715.5285041934287
第4epoch的loss:6.48485460842357,困惑度：655.1437049382483
第5epoch的loss:6.446100002941601,困惑度：630.2395611760187
第6epoch的loss:6.414387935495632,困惑度：610.5669400769976
第7epoch的loss:6.399898208271374,困惑度：601.7837781432316
第8epoch的loss:6.389533061267221,困惑度：595.578416119505
第9epoch的loss:6.371425954704611,困惑度：584.8912629785742
第10epoch的loss:6.355794917325922,困惑度：575.8198879847852
第11epoch的loss:6.358724276904754,困惑度：577.5091445012341
第12epoch的loss:6.3503221465942055,困惑度：572.6771652994742
第13epoch的loss:6.341786946587384,困惑度：567.8100515960152
第14epoch的loss:6.348277413946951,困惑度：571.5073899497354
第15epoch的loss:6.334295928987682,困惑度：563.5724682281186
第16epoch的loss:6.333921603850502,困惑度：563.3615483654044
第17epoch的loss:6.32632978960999,困惑度：559.1008059865
第18epoch的loss:6.3307784544592876,困惑度：561.593598777982
第19epoch的loss:6.324962122827514,困惑度：558.33

In [15]:
def predict_text(text,num,vocab,net):
    net.eval()
    unk_idx=vocab['token_to_idx']['<unk>']
    text=text.split()
    idx_to_token=vocab['idx_to_token']
    token_to_idx=vocab['token_to_idx']
    output_idx=[token_to_idx.get(text[0],unk_idx)]
    state=net.get_new_state(1)
    with torch.no_grad():

        def get_input():
            idx=output_idx[-1]
            return torch.tensor(idx,device=device).reshape(-1,1)
        for token in text[1:]:
            _,state=net(get_input(),state)
            output_idx.append(token_to_idx.get(token,unk_idx))
        for i in range(num):
            y,state=net(get_input(),state)
            y=y[-1].argmax(dim=-1).item()
            output_idx.append(y)
        return ' '.join([idx_to_token[num] for num in output_idx])

In [16]:
predict_text('world',100,vocab,net)

'world and so then i was so for a little or so with a moment as i turned and the machine had been only a minute he said to me it is a little thing who said the psychologist to me you know to me you you said the time traveller it s a thing he said it is an time and we said the medical man with a lamp or a table and only a table he said the time traveller he s all know you said it you said that time has you have been said you you said'